Prompt engineering is about taking a prompt you've written and improving it to get more reliable, higher-quality outputs. This process involves iterative refinement - starting with a basic prompt, evaluating its performance, then systematically applying engineering techniques to improve it.

In [69]:
from google import genai
from dotenv import load_dotenv
import os


load_dotenv(override=True)
API_KEY = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

In [70]:
# Helper functions
def add_user_message(messages, text):
    messages.append({
        "role": "user",
        "parts": [{"text": text}]
    })

def add_model_message(messages, text):
    messages.append({
        "role": "model",
        "parts": [{"text": text}]
    })

def chat(messages, response_schema=None, system_prompt=None, temperature=1.0):
    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=messages,
        config=genai.types.GenerateContentConfig(
            system_instruction=system_prompt,
            max_output_tokens=5000,
            temperature=temperature,
            response_mime_type="application/json" if response_schema else None,
            response_schema=response_schema
        )
    )
    return response.text

In [71]:
# Report Builder
from statistics import mean

def generate_prompt_evaluation_report(evaluation_results):
    total_tests = len(evaluation_results)
    scores = [result["score"] for result in evaluation_results]
    avg_score = mean(scores) if scores else 0
    max_possible_score = 10
    pass_rate = (
        100 * len([s for s in scores if s >= 7]) / total_tests if total_tests else 0
    )

    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Prompt Evaluation Report</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 20px;
                color: #333;
            }}
            .header {{
                background-color: #f0f0f0;
                padding: 20px;
                border-radius: 5px;
                margin-bottom: 20px;
            }}
            .summary-stats {{
                display: flex;
                justify-content: space-between;
                flex-wrap: wrap;
                gap: 10px;
            }}
            .stat-box {{
                background-color: #fff;
                border-radius: 5px;
                padding: 15px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                flex-basis: 30%;
                min-width: 200px;
            }}
            .stat-value {{
                font-size: 24px;
                font-weight: bold;
                margin-top: 5px;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin-top: 20px;
            }}
            th {{
                background-color: #4a4a4a;
                color: white;
                text-align: left;
                padding: 12px;
            }}
            td {{
                padding: 10px;
                border-bottom: 1px solid #ddd;
                vertical-align: top;
            }}
            tr:nth-child(even) {{
                background-color: #f9f9f9;
            }}
            .score {{
                font-weight: bold;
                padding: 5px 10px;
                border-radius: 3px;
                display: inline-block;
            }}
            .score-high {{
                background-color: #c8e6c9;
                color: #2e7d32;
            }}
            .score-medium {{
                background-color: #fff9c4;
                color: #f57f17;
            }}
            .score-low {{
                background-color: #ffcdd2;
                color: #c62828;
            }}
            .output pre {{
                background-color: #f5f5f5;
                border: 1px solid #ddd;
                border-radius: 4px;
                padding: 10px;
                margin: 0;
                font-family: 'Consolas', 'Monaco', 'Courier New', monospace;
                font-size: 14px;
                line-height: 1.4;
                color: #333;
                box-shadow: inset 0 1px 3px rgba(0, 0, 0, 0.1);
                overflow-x: auto;
                white-space: pre-wrap;
                word-wrap: break-word;
            }}
            td {{
                width: 20%;
            }}
            .score-col {{
                width: 80px;
            }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Prompt Evaluation Report</h1>
            <div class="summary-stats">
                <div class="stat-box">
                    <div>Total Test Cases</div>
                    <div class="stat-value">{total_tests}</div>
                </div>
                <div class="stat-box">
                    <div>Average Score</div>
                    <div class="stat-value">{avg_score:.1f} / {max_possible_score}</div>
                </div>
                <div class="stat-box">
                    <div>Pass Rate (≥7)</div>
                    <div class="stat-value">{pass_rate:.1f}%</div>
                </div>
            </div>
        </div>

        <table>
            <thead>
                <tr>
                    <th>Scenario</th>
                    <th>Prompt Inputs</th>
                    <th>Solution Criteria</th>
                    <th>Output</th>
                    <th>Score</th>
                    <th>Reasoning</th>
                </tr>
            </thead>
            <tbody>
    """

    for result in evaluation_results:
        prompt_inputs_html = "<br>".join(
            [
                f"<strong>{key}:</strong> {value}"
                for key, value in result["test_case"]["prompt_inputs"].items()
            ]
        )

        criteria_string = "<br>• ".join(result["test_case"]["solution_criteria"])

        score = result["score"]
        if score >= 8:
            score_class = "score-high"
        elif score <= 5:
            score_class = "score-low"
        else:
            score_class = "score-medium"

        html += f"""
            <tr>
                <td>{result["test_case"]["scenario"]}</td>
                <td class="prompt-inputs">{prompt_inputs_html}</td>
                <td class="criteria">• {criteria_string}</td>
                <td class="output"><pre>{result["output"]}</pre></td>
                <td class="score-col"><span class="score {score_class}">{score}</span></td>
                <td class="reasoning">{result["reasoning"]}</td>
            </tr>
        """

    html += """
            </tbody>
        </table>
    </body>
    </html>
    """

    return html

In [72]:
import json
import re
import concurrent.futures
import time
from textwrap import dedent
from statistics import mean


# Wrapper around chat() that retries on Gemini rate limit errors (429)
# Free tier allows only 5 requests/min so this is essential for larger datasets
def chat_with_retry(messages, response_schema=None, system_prompt=None, temperature=1.0, retries=3):
    for attempt in range(retries):
        try:
            return chat(messages, response_schema, system_prompt, temperature)
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                wait = 60
                print(f"Rate limit hit. Waiting {wait}s... (attempt {attempt+1}/{retries})")
                time.sleep(wait)
            else:
                # Re-raise immediately for non-rate-limit errors
                raise e
    raise Exception("Max retries exceeded")


class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=3):
        # Controls how many API calls run in parallel via ThreadPoolExecutor
        # Keep this low on free tier to avoid rate limits
        self.max_concurrent_tasks = max_concurrent_tasks

    def render(self, template_string, variables):
        # Custom template renderer that replaces {placeholders} with values
        # Uses {{ and }} as escaped braces (like Python f-strings)
        # This avoids conflicts with Gemini JSON schemas that also use curly braces
        placeholders = re.findall(r"{([^{}]+)}", template_string)
        result = template_string
        for placeholder in placeholders:
            if placeholder in variables:
                result = result.replace(
                    "{" + placeholder + "}", str(variables[placeholder])
                )
        return result.replace("{{", "{").replace("}}", "}")

    def generate_unique_ideas(self, task_description, prompt_inputs_spec, num_cases):
        # Step 1 of dataset generation
        # Ask Gemini to brainstorm N distinct testing scenarios before building full test cases
        # Separating idea generation from test case generation gives more diverse results

        prompt = """
        Generate {num_cases} unique, diverse ideas for testing a prompt that accomplishes this task:
        
        <task_description>
        {task_description}
        </task_description>

        The prompt will receive the following inputs:
        <prompt_inputs>
        {prompt_inputs_spec}
        </prompt_inputs>
        
        Each idea should represent a distinct scenario or example that tests different aspects of the task.
        
        Ensure each idea is:
        - Clearly distinct from the others
        - Relevant to the task description
        - Specific enough to guide generation of a full test case
        - Quick to solve without requiring extensive computation or multi-step processing
        - Solvable with no more than 400 tokens of output

        Generate exactly {num_cases} unique ideas.
        """
        # Note: No ```json example block in the prompt
        # Backticks in prompts confuse Gemini when response_mime_type is application/json
        # The response_schema below enforces the structure instead

        system_prompt = "You are a test scenario designer specialized in creating diverse, unique testing scenarios."

        # Build a string showing the expected input keys and their descriptions
        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": str # {val},'

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "task_description": task_description,
                "num_cases": num_cases,
                "prompt_inputs_spec": example_prompt_inputs,
            },
        )

        # Schema enforces Gemini returns a plain array of strings (the ideas)
        schema = {
            "type": "array",
            "items": {"type": "string"}
        }

        messages = []
        add_user_message(messages, rendered_prompt)

        # Removed from original Anthropic version:
        # add_assistant_message(messages, "```json")  -- Gemini does not support prefilling
        # stop_sequences=["```"]                       -- Not needed, schema handles JSON output

        text = chat_with_retry(
            messages,
            response_schema=schema,
            system_prompt=system_prompt,
            temperature=1.0,   # High temperature for diverse, creative ideas
        )
        return json.loads(text)

    def generate_test_case(self, task_description, idea, prompt_inputs_spec={}):
        # Step 2 of dataset generation
        # Takes one idea from generate_unique_ideas and builds a full test case from it
        # Runs concurrently for all ideas via ThreadPoolExecutor in generate_dataset()

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": "EXAMPLE_VALUE", // {val}\n'

        # Build comma-separated list of allowed keys to inject into the prompt
        allowed_keys = ", ".join([f'"{key}"' for key in prompt_inputs_spec.keys()])

        prompt = """
        Generate a single detailed test case for a prompt evaluation based on:
        
        <task_description>
        {task_description}
        </task_description>
        
        <specific_idea>
        {idea}
        </specific_idea>
        
        <allowed_input_keys>
        {allowed_keys}
        </allowed_input_keys>
        
        IMPORTANT REQUIREMENTS:
        - You MUST ONLY use these exact input keys in your prompt_inputs: {allowed_keys}        
        - Do NOT add any additional keys to prompt_inputs
        - All keys listed in allowed_input_keys must be included in your response
        - Make the test case realistic and practically useful
        - Include measurable, concise solution criteria
        - The solution criteria should ONLY address the direct requirements of the task description
        - Keep solution criteria simple, focused, and directly tied to the fundamental task
        - The test case should be tailored to the specific idea provided
        - Quick to solve without requiring extensive computation or multi-step processing
        - Solvable with no more than 400 tokens of output
        - DO NOT include any fields beyond those specified
        """
        # Note: Removed the ```json example block from the original
        # Backticks in prompts break Gemini JSON output even with response_mime_type set
        # The response_schema below fully enforces the output structure

        system_prompt = "You are a test case creator specializing in designing evaluation scenarios."

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "allowed_keys": allowed_keys,
                "task_description": task_description,
                "idea": idea,
                "example_prompt_inputs": example_prompt_inputs,
            },
        )

        # Schema enforces the exact structure of each test case
        # prompt_inputs keys are built dynamically from prompt_inputs_spec
        # so the schema always matches the expected input keys exactly
        schema = {
            "type": "object",
            "properties": {
                "prompt_inputs": {
                    "type": "object",
                    "properties": {k: {"type": "string"} for k in prompt_inputs_spec.keys()}
                },
                "solution_criteria": {
                    "type": "array",
                    "items": {"type": "string"}
                }
            },
            "required": ["prompt_inputs", "solution_criteria"]
        }

        messages = []
        add_user_message(messages, rendered_prompt)

        # Removed from original Anthropic version:
        # add_assistant_message(messages, "```json")  -- Gemini does not support prefilling
        # stop_sequences=["```"]                       -- Not needed, schema handles JSON output

        text = chat_with_retry(
            messages,
            response_schema=schema,
            system_prompt=system_prompt,
            temperature=0.7,   # Slightly lower than ideas — we want realistic, not too creative
        )

        test_case = json.loads(text)

        # Attach metadata not in the schema — these are used later by the report builder
        test_case["task_description"] = task_description
        test_case["scenario"] = idea
        return test_case

    def generate_dataset(
        self,
        task_description,
        prompt_inputs_spec={},
        num_cases=1,
        output_file="dataset.json",
    ):
        # Orchestrates the full dataset generation pipeline:
        # 1. Generate N ideas
        # 2. For each idea, generate a full test case (in parallel)
        # 3. Save to JSON file

        ideas = self.generate_unique_ideas(
            task_description, prompt_inputs_spec, num_cases
        )

        dataset = []
        completed = 0
        total = len(ideas)
        last_reported_percentage = 0

        # ThreadPoolExecutor runs generate_test_case() for all ideas concurrently
        # max_workers is set at class init -- keep low on free tier
        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_idea = {
                executor.submit(
                    self.generate_test_case,
                    task_description,
                    idea,
                    prompt_inputs_spec,
                ): idea
                for idea in ideas
            }

            for future in concurrent.futures.as_completed(future_to_idea):
                try:
                    result = future.result()
                    completed += 1
                    current_percentage = int((completed / total) * 100)
                    milestone_percentage = (current_percentage // 20) * 20

                    # Only print progress at 20%, 40%, 60%, 80%, 100% milestones
                    if milestone_percentage > last_reported_percentage:
                        print(f"Generated {completed}/{total} test cases")
                        last_reported_percentage = milestone_percentage

                    dataset.append(result)
                except Exception as e:
                    print(f"Error generating test case: {e}")

        # Save dataset to disk so it can be reused across eval runs
        # without regenerating (saves API calls and time)
        with open(output_file, "w") as f:
            json.dump(dataset, f, indent=2)

        return dataset

    def grade_output(self, test_case, output, extra_criteria):
        # Uses Gemini as a judge to score how well the output solves the task
        # Returns a structured dict with strengths, weaknesses, reasoning, and score

        # Format prompt_inputs as a JSON-like string for the eval prompt
        prompt_inputs = ""
        for key, value in test_case["prompt_inputs"].items():
            val = value.replace("\n", "\\n")
            prompt_inputs += f'"{key}":"{val}",\n'

        # extra_criteria is optional -- if provided, any violation auto-fails the output
        extra_criteria_section = ""
        if extra_criteria:
            extra_criteria_template = """
            Mandatory Requirements - ANY VIOLATION MEANS AUTOMATIC FAILURE (score of 3 or lower):
            <extra_important_criteria>
            {extra_criteria}
            </extra_important_criteria>
            """
            extra_criteria_section = self.render(
                dedent(extra_criteria_template),
                {"extra_criteria": extra_criteria},
            )

        eval_template = """
        Your task is to evaluate the following AI-generated solution with EXTREME RIGOR.

        Original task description:
        <task_description>
        {task_description}
        </task_description>

        Original task inputs:
        <task_inputs>
        {{ {prompt_inputs} }}
        </task_inputs>

        Solution to Evaluate:
        <solution>
        {output}
        </solution>

        Criteria you should use to evaluate the solution:
        <criteria>
        {solution_criteria}
        </criteria>

        {extra_criteria_section}

        Scoring Guidelines:
        * Score 1-3: Solution fails to meet one or more MANDATORY requirements
        * Score 4-6: Solution meets all mandatory requirements but has significant deficiencies
        * Score 7-8: Solution meets all mandatory requirements and most secondary criteria
        * Score 9-10: Solution meets all mandatory and secondary criteria perfectly

        IMPORTANT SCORING INSTRUCTIONS:
        * Grade the output based ONLY on the listed criteria
        * If a solution meets all criteria give it a 10
        * ANY violation of a mandatory requirement MUST result in a score of 3 or lower
        * The full 1-10 scale should be utilized

        Keep your response concise and direct.
        """
        # Note: Removed the ```json example block from the original
        # Backticks in prompts break Gemini JSON output even with response_mime_type set

        eval_prompt = self.render(
            dedent(eval_template),
            {
                "task_description": test_case["task_description"],
                "prompt_inputs": prompt_inputs,
                "output": output,
                "solution_criteria": "\n".join(test_case["solution_criteria"]),
                "extra_criteria_section": extra_criteria_section,
            },
        )

        # Schema enforces the exact grading structure Gemini must return
        eval_schema = {
            "type": "object",
            "properties": {
                "strengths":  {"type": "array", "items": {"type": "string"}},
                "weaknesses": {"type": "array", "items": {"type": "string"}},
                "reasoning":  {"type": "string"},
                "score":      {"type": "integer"}
            },
            "required": ["strengths", "weaknesses", "reasoning", "score"]
        }

        messages = []
        add_user_message(messages, eval_prompt)

        # Removed from original Anthropic version:
        # add_assistant_message(messages, "```json")  -- Gemini does not support prefilling
        # stop_sequences=["```"]                       -- Not needed, schema handles JSON output

        # temperature=0.0 for deterministic, consistent grading across runs
        eval_text = chat_with_retry(
            messages,
            response_schema=eval_schema,
            temperature=0.0,
        )
        return json.loads(eval_text)

    def run_test_case(self, test_case, run_prompt_function, extra_criteria=None):
        # Runs a single test case end to end:
        # 1. Calls run_prompt_function with the test inputs to get the model output
        # 2. Grades that output against the solution criteria
        # Returns a result dict used by run_evaluation() and the report builder

        output = run_prompt_function(test_case["prompt_inputs"])
        model_grade = self.grade_output(test_case, output, extra_criteria)
        return {
            "output":    output,
            "test_case": test_case,
            "score":     model_grade["score"],
            "reasoning": model_grade["reasoning"],
        }

    def run_evaluation(
        self,
        run_prompt_function,
        dataset_file,
        extra_criteria=None,
        json_output_file="output.json",
        html_output_file="output.html",
    ):
        # Main entry point for running a full evaluation
        # Loads dataset from file, runs all test cases in parallel,
        # prints average score, and saves both JSON and HTML reports

        with open(dataset_file, "r") as f:
            dataset = json.load(f)

        results = []
        completed = 0
        total = len(dataset)
        last_reported_percentage = 0

        # Run all test cases concurrently to speed up evaluation
        # max_workers controls parallelism -- keep low on free tier
        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_test_case = {
                executor.submit(
                    self.run_test_case,
                    test_case,
                    run_prompt_function,
                    extra_criteria,
                ): test_case
                for test_case in dataset
            }

            for future in concurrent.futures.as_completed(future_to_test_case):
                result = future.result()
                completed += 1
                current_percentage = int((completed / total) * 100)
                milestone_percentage = (current_percentage // 20) * 20

                # Only print progress at 20%, 40%, 60%, 80%, 100% milestones
                if milestone_percentage > last_reported_percentage:
                    print(f"Graded {completed}/{total} test cases")
                    last_reported_percentage = milestone_percentage
                results.append(result)

        average_score = mean([result["score"] for result in results])
        print(f"Average score: {average_score}")

        # Save raw results as JSON for programmatic use
        with open(json_output_file, "w") as f:
            json.dump(results, f, indent=2)

        # Save formatted HTML report — open in browser to view scores, reasoning, outputs
        html = generate_prompt_evaluation_report(results)
        with open(html_output_file, "w", encoding="utf-8") as f:
            f.write(html)

        return results

In [73]:
evaluator = PromptEvaluator(max_concurrent_tasks=3)

In [6]:
dataset = evaluator.generate_dataset(
    task_description= "Write a compact , concise 1 day meal plan for single athlete",

    prompt_inputs_spec={
        "height": "Athletes height in cm",
        "weight": "Athletes weight in kg",
        "goal": "Goal of the athlete",
        "restrictions":"Dietary restrictions of the athlete",
    },
    
    output_file= "dataset.json",
    num_cases=3
)

Generated 1/3 test cases
Generated 2/3 test cases
Generated 3/3 test cases


In [ ]:
def run_prompt(prompt_inputs):
    prompt = f"""
    What should this person eat:

    -Height: {prompt_inputs["height"]}
    -Weight: {prompt_inputs["weight"]}
    -Goal: {prompt_inputs["goal"]}
    -Dietary Restrictions: {prompt_inputs["restrictions"]}
    """

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

In [10]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt, dataset_file="dataset.json"
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 5


When crafting that crucial first line, you want to focus on two key principles: clarity and directness. This means using simple language that leaves no room for ambiguity about what you want Claude to do.

In [42]:
def run_prompt(prompt_inputs):
    prompt = f"""
    Generate a one day meal plan an athelete that meets their dietary restrictions.

    -Height: {prompt_inputs["height"]}
    -Weight: {prompt_inputs["weight"]}
    -Goal: {prompt_inputs["goal"]}
    -Dietary Restrictions: {prompt_inputs["restrictions"]}
    """

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

In [43]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt, dataset_file="dataset.json",
    extra_criteria="""
     The output should include:
     - Daily caloric breakdown
     - Micronutrient Breakdown
     - Meals with exact foods, portions and timing
    """
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 3


When working with Claude, one of the most effective ways to improve your results is to be specific about what you want. Instead of leaving everything up to the model's interpretation, you can provide clear guidelines or steps that direct Claude toward the kind of output you're looking for.

Think about it this way: if you ask Claude to "write a short story about a character who discovers a hidden talent," Claude could go in countless directions. The story might be 200 words or 2,000 words. It might have one character or five. It could focus on any type of talent discovery scenario.

In [44]:
def run_prompt(prompt_inputs):
    prompt = f"""
    Generate a one day meal plan an athelete that meets their dietary restrictions.

    -Height: {prompt_inputs["height"]}
    -Weight: {prompt_inputs["weight"]}
    -Goal: {prompt_inputs["goal"]}
    -Dietary Restrictions: {prompt_inputs["restrictions"]}

    Guidelines:
    1. Include accurate daily calorie amount

    2. Show protein, fat, and carb amounts

    3. Specify when to eat each meal

    4. Use only foods that fit restrictions

    5. List all portion sizes in grams

    6. Keep budget-friendly if mentioned
    """

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

In [45]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt, dataset_file="dataset.json",
    extra_criteria="""
     The output should include:
     - Daily caloric breakdown
     - Micronutrient Breakdown
     - Meals with exact foods, portions and timing
    """
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 1


When you're building prompts that include a lot of content, Claude can sometimes struggle to understand which pieces of text belong together or what different sections are supposed to represent. XML tags provide a simple way to add structure and clarity to your prompts, especially when you're interpolating large amounts of data.

In [74]:
def run_prompt(prompt_inputs):
    prompt = f"""
    Generate a one day meal plan an athelete that meets their dietary restrictions.

    <athlete_information>
    -Height: {prompt_inputs["height"]}
    -Weight: {prompt_inputs["weight"]}
    -Goal: {prompt_inputs["goal"]}
    -Dietary Restrictions: {prompt_inputs["restrictions"]}
    </athlete_information>
    Guidelines:
    1. Include accurate daily calorie amount

    2. Show protein, fat, and carb amounts

    3. Specify when to eat each meal

    4. Use only foods that fit restrictions

    5. List all portion sizes in grams

    6. Keep budget-friendly if mentioned
    """

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

In [75]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt, dataset_file="dataset.json",
    extra_criteria="""
     The output should include:
     - Daily caloric breakdown
     - Micronutrient Breakdown
     - Meals with exact foods, portions and timing
    """
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 3


Providing examples in your prompts is one of the most effective prompt engineering techniques you'll use. This approach, known as "one-shot" or "multi-shot" prompting, involves giving Claude sample input/output pairs to guide its responses.

In [65]:
# Define and run the prompt you want to evaluate, returning the raw model output
# This function is executed once for each test case
def run_prompt(prompt_inputs):
    prompt = f"""
    Generate a one-day meal plan for an athlete that meets their dietary restrictions.

    <athlete_information> 
    - Height: {prompt_inputs["height"]} 
    - Weight: {prompt_inputs["weight"]} 
    - Goal: {prompt_inputs["goal"]} 
    - Dietary restrictions: {prompt_inputs["restrictions"]} 
    </athlete_information>

    Guidelines:
    1. Include accurate daily calorie amount
    2. Show protein, fat, and carb amounts
    3. Specify when to eat each meal
    4. Use only foods that fit restrictions
    5. List all portion sizes in grams
    6. Keep budget-friendly if mentioned

    Here is an example with a sample input and an ideal output:
    <sample_input>
    height: 170
    weight: 70
    goal: Maintain fitness and improve cholesterol levels
    restrictions: High cholesterol
    </sample_input>
    <ideal_output>
    Here is a one-day meal plan for an athlete aiming to maintain fitness and improve cholesterol levels:

    *   **Calorie Target:** Approximately 2500 calories
    *   **Macronutrient Breakdown:** Protein (140g), Fat (70g), Carbs (340g)

    **Meal Plan:**

    *   **Breakfast (7:00 AM):** Oatmeal (80g dry weight) with berries (100g) and walnuts (15g). Skim milk (240g).
        *   Protein: 15g, Fat: 15g, Carbs: 60g
    *   **Mid-Morning Snack (10:00 AM):** Apple (150g) with almond butter (30g).
        *   Protein: 7g, Fat: 18g, Carbs: 25g
    *   **Lunch (1:00 PM):** Grilled chicken breast (120g) salad with mixed greens (150g), cucumber (50g), tomato (50g), and a light vinaigrette dressing (30g). Whole wheat bread (60g).
        *   Protein: 40g, Fat: 15g, Carbs: 70g
    *   **Afternoon Snack (4:00 PM):** Greek yogurt (170g, non-fat) with a banana (120g).
        *   Protein: 20g, Fat: 0g, Carbs: 40g
    *   **Dinner (7:00 PM):** Baked salmon (140g) with steamed broccoli (200g) and quinoa (75g dry weight).
        *   Protein: 40g, Fat: 20g, Carbs: 80g
    *   **Evening Snack (9:00 PM):** Small handful of almonds (20g).
        *   Protein: 8g, Fat: 12g, Carbs: 15g

    This meal plan prioritizes lean protein sources, whole grains, fruits, and vegetables, while limiting saturated and trans fats to support healthy cholesterol levels.
    </ideal_output>
    This example meal plan is well-structured, provides detailed information on food choices and quantities, and aligns with the athlete's goals and restrictions.
    """

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

In [66]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions, and timing
    """,
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 9
